<a href="https://colab.research.google.com/github/AliyaBadmaeva/PDP/blob/main/Badmaeva_AA_RuBert_PDP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datetime import datetime
current_datetime = datetime.now()
print(current_datetime)

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
from tqdm import tqdm

In [ ]:
#!pip uninstall transformers

In [ ]:
#!pip install transformers

In [ ]:
!pip show transformers

In [ ]:
import warnings
# игнорируем предупреждения
warnings.filterwarnings('ignore')

In [ ]:
#!pip install openpyxl  # установка бибилиотеки для чтения Excel

In [ ]:
DATASET = 'stepik_all_reviews_2025-09-07_05-10-59.xlsx'

In [ ]:
df = pd.read_excel(DATASET) # читаем датасет из отзывов со Степика

In [ ]:
df.head(10)  # Первые 10 строк датасета

In [ ]:
df.info()

In [ ]:
df.describe()  # статистические данные

In [ ]:
df.rename(columns={'Звёзды': 'label', 'Отзыв' :'text'}, inplace=True)  # заменяем названия на кириллице

In [ ]:
del df['Курс']  # удаляем лишние признаки

In [ ]:
del df['ID отзыва']  # удаляем ID отзыва

In [ ]:
df['label'].value_counts()  # количество данных в целевом признаке

In [ ]:
df[df['label']==0]  # отзывы, имеющие 0 баллов

Отзыв в ячейке выше не удалось спарсить, а пустые отзывы не несут никакой пользы, удалим их.

In [ ]:
df = df[df['label'] != 0]  # перезапишем датасет, исключим отзывы, имеющие 0 баллов по фильтру

In [ ]:
df['label'].value_counts()

Преобразуем значения в 3 тональности: положительная, нейтральная и отрицательная, где 1,2 балла - отрицательная, 3 балла - нейтральная, а 4 и 5 баллов - положительная тональность отзывов.

In [ ]:
df["label"] = df["label"].map({1:0, 2:0, 3:1, 4:2, 5:2})  # три тональности

In [ ]:
df['label'].value_counts()

In [ ]:
# удалим нулевые значения по наличию в столбце отзыв, чтобы не было потом проблем с созданием тензоров
df.dropna(axis=0, how='any',subset=['text'], inplace=True)

In [ ]:
df.duplicated().sum()  # количество дублей

In [ ]:
df = df.drop_duplicates(subset=['text'])  # удалим дибли по столбцу с текстом отзыва

In [ ]:
df.describe()

In [ ]:
df = df[df['text'].str.count(r'[А-Яа-я]') >= 2].reset_index(drop=True)  # Оставим только отзывы, имеющие кириллицу и по длине от 2 букв

In [ ]:
def clean_text(t):  # Очистим текст
    t = str(t).strip()
    if len(t) < 10:           # короткие отзывы неинформативны
        return ""
    if t.lower() in {"ок", "none", "-", "+"}:
        return ""
    return t

df["text"] = df["text"].apply(clean_text)
df = df[df["text"] != ""].reset_index(drop=True)

In [ ]:
print(df.shape)          # сколько осталось (строки, столбцы)
print(df['text'].head()) # примеры

In [ ]:
def balanced_dataset_len(df: pd.DataFrame,
                         text_col: str = 'text',
                         label_col: str = 'label',
                         n_per_class: int = 2000,
                         seed: int = 42) -> pd.DataFrame:
    """
    Балансировка по label и равномерное распределение по длине текста
    (квантили 0-33 %, 33-66 %, 66-100 %)
    """
    df = df.copy()
    df['length'] = df[text_col].str.len()

    def sample_quantiles(x):
        if len(x) <= n_per_class // 3:          # мало данных – берём всё
            return x
        # делим на 3 квантиля по длине
        x = x.sort_values('length')
        q1, q2 = x['length'].quantile([0.33, 0.66])
        parts = [x[x['length'] <= q1],
                 x[(x['length'] > q1) & (x['length'] <= q2)],
                 x[x['length'] > q2]]
        # из каждого квантиля берём поровну
        n_from_part = n_per_class // 3
        return pd.concat([p.sample(min(len(p), n_from_part), random_state=seed)
                          for p in parts], ignore_index=True)

    out = (df.groupby(label_col, group_keys=False)
             .apply(sample_quantiles)
             .reset_index(drop=True))

    # если вдруг набралось чуть больше – обрезаем до ровного числа
    out = (out.groupby(label_col, group_keys=False)
             .apply(lambda x: x.sample(min(len(x), n_per_class), random_state=seed))
             .reset_index(drop=True))

    return out.drop(columns='length')


# использование
df_bal = balanced_dataset_len(df, text_col='text', label_col='label', n_per_class=2000)
print(df_bal['label'].value_counts())
print(df_bal['text'].str.len().describe())   # проверим распределение длин

In [ ]:
"""import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "sentencepiece"])"""

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.15,          # 15 % на валидацию
    stratify=df['label'],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print('Train:', train_df['label'].value_counts())
print('Val  :', val_df['label'].value_counts())

In [ ]:
X_train = train_df.text
X_val = val_df.text

y_train = train_df.label
y_val = val_df.label

In [ ]:
! nvidia-smi

In [ ]:
MAX_LEN = 256

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
#!pip install torch

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

In [ ]:
base = "DeepPavlov/rubert-base-cased"   # без головы
tokenizer = AutoTokenizer.from_pretrained(base)


In [ ]:
tokenizer.save_pretrained('.')

In [ ]:
from transformers import DataCollatorWithPadding
from datasets import Dataset

In [ ]:
def tok_func(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

train_ds = train_df[["text", "label"]].rename(columns={"label": "labels"})
val_ds = val_df[["text", "label"]].rename(columns={"label": "labels"})

train_ds = Dataset.from_pandas(train_ds)
val_ds = Dataset.from_pandas(val_ds)

# 2. Токенизация на лету + dynamic padding
train_ds = train_ds.map(tok_func, batched=True)
val_ds = val_ds.map(tok_func, batched=True)

# 3. DataCollator делает паддинг «по максимуму внутри батча»
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
#!pip install --upgrade nbformat
#!pip install --upgrade nbconvert

In [ ]:
from transformers import Trainer, TrainingArguments

In [ ]:
#!pip install evaluate

In [ ]:
id2label = {0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"}
label2id = {"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
            base,
            num_labels=3,          # 3 тональности
            problem_type="single_label_classification",
            id2label=id2label,
            label2id=label2id).to("cuda")

In [ ]:
from torch.optim import AdamW

loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
    }

In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./ruBERT_sentiment",
    overwrite_output_dir=True,
    num_train_epochs=8,                 # 8 эпох
    per_device_train_batch_size=4,      #  batch на GPU
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=8,      # эффективный batch = 8×4 = 32
    learning_rate=1e-5,                 # lr для AdamW
    weight_decay=0.02,                  # L2-регуляризация
    warmup_ratio=0.15,                   # 10 % шагов – линейный разогрев
    lr_scheduler_type="cosine",         # косинусное затухание lr
    bf16=False,                         # у RTX 3070 нет bf16, поэтому
    fp16=True,                          # экономия 30 % памяти + скорость
    dataloader_num_workers=4,           # быстрее загрузка
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="accuracy",         # следим за accuracy
    load_best_model_at_end=True,
    report_to="none",                   # не шлём логи в wandb
    greater_is_better=True,
    save_total_limit=2,                 # храним 2 лучших чек-поинта
)
model.gradient_checkpointing_enable()

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
# train the model
trainer.train()

In [ ]:
# evaluate the current model after training
trainer.evaluate()

In [ ]:
# !pip install 'accelerate>=0.26.0'

In [ ]:
from transformers import pipeline

In [ ]:
trainer.save_model(output_dir='./ruBert_results')

In [ ]:
text = """
Мне не нравится эта дисциплина. Мне больше по душе такие дисциплины, как МО, например.
"""


In [ ]:
classifier = pipeline("sentiment-analysis", model="./ruBert_results")
classifier(text)

In [ ]:
current_datetime = datetime.now()
print(current_datetime)

In [ ]:
import json, io, os, glob

# 1. находим .ipynb в текущей папке
files = glob.glob('*.ipynb')
if not files:
    raise RuntimeError('Нет .ipynb в папке')
nb = files[0]                                   # берём первый

# 2. читаем JSON
with io.open(nb, 'r', encoding='utf-8') as f:
    data = json.load(f)

# 3. УДАЛЯЕМ widgets везде
for cell in data.get('cells', []):
    cell.get('metadata', {}).pop('widgets', None)

# 4. перезаписываем
with io.open(nb, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=1, ensure_ascii=False)

print('widgets удалены')

In [ ]:
!jupyter nbconvert "$nb" --to html